# Project 1 — Issue Report Classification
## Notebook 05: Final comparison and error analysis

Pulls every saved result into one table, compares against the published SetFit
baseline, and analyses where the best model fails.

All results are read from `results/tables/*.json`, written by
`scripts/run_experiments.py`. Nothing is re-trained here, so the report can be
rebuilt without a GPU and the numbers cannot drift from what the code
produced.

In [1]:
import json

import pandas as pd

from ai4se import error_analysis as ea
from ai4se.classical import TUNED_PREPROCESSING, tuned_logistic_regression
from ai4se.evaluation import SETFIT_OVERALL, leaderboard_from_disk, to_latex
from ai4se.loader import PROJECT_ROOT, load_split
from ai4se.preprocessing import make_cleaner

pd.set_option("display.width", 160)
TABLES = PROJECT_ROOT / "results" / "tables"

board = leaderboard_from_disk(TABLES)
board

,react,tensorflow,vscode,bitcoin,opencv,overall,AUC,vs SetFit
model,,,,,,,,
SetFit (NLBSE'24 baseline),0.8718,0.8644,0.8262,0.7555,0.8173,0.8270,NaN,0.0000
Ensemble (SetFit + TF-IDF + MPNet),0.8505,0.8520,0.7957,0.7691,0.8168,0.8168,0.9319,-0.0102
SetFit (MPNet),0.8438,0.8710,0.8104,0.7488,0.7771,0.8102,0.9190,-0.0168
Ensemble (SetFit + TF-IDF),0.8396,0.8521,0.7922,0.7459,0.7966,0.8053,0.9244,-0.0217
SetFit (reproduction),0.8322,0.8414,0.7816,0.7464,0.7896,0.7982,0.9181,-0.0288
Ensemble (TF-IDF + MPNet + CNN),0.8296,0.8395,0.7494,0.7694,0.7665,0.7909,0.9245,-0.0361
Ensemble (TF-IDF + MPNet),0.8147,0.7856,0.7762,0.7497,0.7925,0.7838,0.9198,-0.0432
"SetFit (MiniLM, matched settings)",0.7934,0.8618,0.7624,0.7170,0.7735,0.7816,0.9144,-0.0454
TF-IDF + Logistic Regression (tuned),0.8334,0.8155,0.7234,0.6793,0.7496,0.7603,0.9018,-0.0667


---
## 1. Reading the leaderboard

| Group | Best result | Gap to baseline |
|---|---|---|
| Trivial floors | 0.5395 (keyword rules) | −0.288 |
| Classical ML (Track B) | 0.7603 | −0.067 |
| Neural (Track C) | 0.7438 | −0.083 |
| Frozen embeddings | 0.7557 (MPNet) | −0.071 |
| SetFit reproduction | 0.8102 (MPNet) | −0.017 |
| **Ensembles** | **0.8168** | **−0.010** |
| **SetFit (published baseline)** | **0.8270** | — |

The best result, a soft-voting ensemble of SetFit, TF-IDF and frozen MPNet,
closes **98.5%** of the distance from a majority classifier to the baseline.
On `bitcoin/bitcoin` it **beats** the baseline (0.7691 vs 0.7555), and SetFit
on MPNet beats it on `tensorflow` (0.8710 vs 0.8644).

### The encoder effect, measured three times

| Stage | Conclusion | Status |
|---|---|---|
| Projected additively | SetFit-MPNet ≈ 0.8482, beats baseline | **wrong by 0.038** |
| Measured, uncontrolled | encoder worth +0.0120 after fine-tuning | **confounded** |
| Measured, matched settings | encoder worth **+0.0286** | holds |

The uncontrolled comparison ran MiniLM at `batch_size=16, max_seq_length=256`
and MPNet at 8 and 128 — MPNet was handicapped. Re-running MiniLM at MPNet's
settings gives 0.7816, so **the reduced settings alone cost 0.0166**.

| | MiniLM | MPNet | encoder effect |
|---|---|---|---|
| Frozen + LogReg | 0.7057 | 0.7557 | **+0.0500** |
| SetFit @ batch 8, seq 128 | 0.7816 | 0.8102 | **+0.0286** |

Sub-additivity is real but mild: the encoder keeps **57%** of its frozen value
after fine-tuning, not the 24% the uncontrolled numbers implied.

Two methodological points, in order of importance:

1. Effects measured separately were assumed to compose, and they did not.
2. The first correction was itself overstated, because the comparison behind it
   was confounded. Only the matched re-run settled it.

**Training settings matter about as much as encoder size here** — halving batch
and sequence length cost 0.0166 against +0.0286 for doubling encoder depth and
width. A result quoted without them is not comparable with another.

In [2]:
repos = ["react", "tensorflow", "vscode", "bitcoin", "opencv"]
display(board.loc[:, repos].describe().loc[["mean", "std", "min", "max"]].round(4))
print("\nhardest and easiest project for each model (top 6):")
subset = board.loc[:, repos].head(6)
display(pd.DataFrame({
    "easiest": subset.idxmax(axis=1),
    "hardest": subset.idxmin(axis=1),
    "spread": (subset.max(axis=1) - subset.min(axis=1)).round(4),
}))

,react,tensorflow,vscode,bitcoin,opencv
mean,0.7511,0.7465,0.6816,0.6562,0.6891
std,0.1703,0.1740,0.1564,0.1438,0.1569
min,0.1667,0.1667,0.1667,0.1667,0.1667
max,0.8718,0.8710,0.8262,0.7694,0.8173



hardest and easiest project for each model (top 6):


,easiest,hardest,spread
model,,,
SetFit (NLBSE'24 baseline),react,bitcoin,0.1163
Ensemble (SetFit + TF-IDF + MPNet),tensorflow,bitcoin,0.0829
SetFit (MPNet),tensorflow,bitcoin,0.1222
Ensemble (SetFit + TF-IDF),tensorflow,bitcoin,0.1062
SetFit (reproduction),tensorflow,bitcoin,0.0950
Ensemble (TF-IDF + MPNet + CNN),tensorflow,vscode,0.0901


`bitcoin/bitcoin` is hardest for almost every model, and for the published
baseline too (0.7555, its lowest). Project difficulty is a property of the data
rather than of any approach.

Two exceptions are worth reporting. The SetFit + TF-IDF + MPNet ensemble
reaches **0.7691 on `bitcoin`** and the GPU-free TF-IDF + MPNet + CNN ensemble
**0.7694**, both above the baseline's 0.7555. SetFit on MPNet reaches **0.8710
on `tensorflow`** against 0.8644. Those are the only places anything built here
beats the baseline — and two of the three are ensembles, on the hardest
project.

In [3]:
train = load_split("train", kind="memory")
test = load_split("test", kind="memory")
train.apply(make_cleaner(**TUNED_PREPROCESSING))
test.apply(make_cleaner(**TUNED_PREPROCESSING))

predictions = ea.collect_predictions(tuned_logistic_regression, train, test)
display(ea.per_repository_scores(predictions))

,F1 bug,F1 feature,F1 question,F1 avg,errors
repository,,,,,
bitcoin/bitcoin,0.6374,0.7586,0.6419,0.6793,96
facebook/react,0.9175,0.8302,0.7526,0.8334,50
microsoft/vscode,0.6957,0.7117,0.7629,0.7234,83
opencv/opencv,0.6630,0.8195,0.7664,0.7496,74
tensorflow/tensorflow,0.8290,0.8526,0.7650,0.8155,56


In [4]:
ea.confusion_summary(predictions)

,true,predicted,count,share of errors %,share of all %
0,bug,question,96,26.7,6.4
1,question,feature,70,19.5,4.7
2,feature,question,57,15.9,3.8
3,bug,feature,53,14.8,3.5
4,question,bug,49,13.6,3.3
5,feature,bug,34,9.5,2.3


**`bug` misclassified as `question` is the single biggest error type**,
accounting for over a quarter of all mistakes. The two classes genuinely
overlap: a user who is unsure whether behaviour is broken writes something that
reads as both a bug report and a question, and the GitHub label they chose
reflects a maintainer's judgement rather than anything in the text.

The three `question` confusions together account for a large share of errors,
which is consistent with `question` having the lowest per-class F1 throughout.

### Does length matter?

In [5]:
ea.errors_by_length(predictions, bins=5)

,issues,accuracy,median_words
bucket,,,
"(0.999, 69.8]",300,0.7467,41.0
"(69.8, 127.0]",307,0.7655,103.0
"(127.0, 178.0]",297,0.8148,152.0
"(178.0, 280.2]",296,0.7601,218.5
"(280.2, 5050.0]",300,0.7167,394.0


Accuracy peaks in the middle of the length distribution and falls at both ends.
Very short issues carry too little text to classify; very long ones are diluted,
and truncation discards part of them. This justifies reporting the truncation
threshold as a hyperparameter rather than an implementation detail.

### What misleads the model?

In [6]:
display(ea.misleading_terms(predictions, "question", "bug", n=10))
display(ea.misleading_terms(predictions, "bug", "question", n=10))

,term,in_errors_%,in_correct_%,lift
0,"(videos,",28.6,1.6,15.59
1,etc),28.6,1.6,15.59
2,"forum.opencv.org,",32.7,1.8,15.59
3,"overflow,",32.7,1.8,15.59
4,bar,6.1,0.3,11.69
5,gcc,18.4,1.6,10.02
6,solution,32.7,3.1,9.59
7,follow,6.1,0.5,7.80
8,detected,6.1,0.5,7.80
9,files,32.7,4.2,7.34


,term,in_errors_%,in_correct_%,lift
0,converted,5.2,0.0,18.33
1,distribution,5.2,0.0,18.33
2,its,5.2,0.0,18.33
3,inside,5.2,0.0,18.33
4,**to,14.6,0.6,17.11
5,reproduce**,14.6,0.6,17.11
6,behavior**,17.7,0.9,15.58
7,**actual,17.7,0.9,15.58
8,**expected,17.7,0.9,15.58
9,necessary,4.2,0.0,14.67


The `question → bug` table is dominated by fragments of OpenCV's issue
template. Project-specific boilerplate is correlated with the class in the
training data, so the model learns the template rather than the content. This is
a concrete argument for more aggressive template stripping — and a reminder that
the per-project protocol lets each classifier overfit to its own project's
conventions.

### The most confident mistakes

Confident errors are the informative ones. A near-tie means a genuinely
ambiguous issue; a confident mistake means the model learned something wrong, or
the ground-truth label is questionable.

In [7]:
mistakes = ea.worst_mistakes(predictions, n=12)
ea.mistakes_frame(mistakes)[["repository", "true", "predicted", "confidence", "title"]]

,repository,true,predicted,confidence,title
0,facebook/react,feature,bug,0.988022,[DevTools] Manifest version 2 is deprecated
1,microsoft/vscode,bug,question,0.979604,[Bug] Tela preta no terminal do VS Code
2,bitcoin/bitcoin,question,feature,0.969963,`sendtoaddress` tries to use unspendable UTXOs...
3,microsoft/vscode,question,feature,0.963986,Allow webview context menus triggered by prima...
4,microsoft/vscode,bug,question,0.960837,Forders not opening
5,tensorflow/tensorflow,question,bug,0.957058,Error when runnning tensorflow.python.ops.spar...
6,opencv/opencv,feature,question,0.939564,How to use opencv to erase text from images an...
7,opencv/opencv,question,bug,0.932370,ANN_MLP - large dataset results in stack overf...
8,bitcoin/bitcoin,question,bug,0.930013,Intermittent issue in p2p_ibd_stalling.py ...
9,microsoft/vscode,question,feature,0.928956,please allow for multiple tunnels on same machine


Rather than eyeball these, quantify what is going on. Many titles carry an
explicit self-declaration of their type — `[Feature Request]`, `How to ...`,
`[Bug]`, `... fails` — which can be checked against the ground-truth label.

In [8]:
from collections import Counter

worst = json.loads((TABLES / "error_analysis.json").read_text())["worst"]

markers = {
    "feature": ["feature request", "[feature]", "allow ", "add support", "please add"],
    "bug": ["[bug]", "bug:", "crash", "error", "fails", "broken"],
    "question": ["how to", "how do", "what is", "why "],
}

conflicts = []
for mistake in worst:
    title = mistake["title"].lower()
    for declared, patterns in markers.items():
        if declared != mistake["true"] and any(p in title for p in patterns):
            conflicts.append({**mistake, "title_declares": declared})
            break

print("true label of the 25 most confident mistakes:")
for label, count in Counter(m["true"] for m in worst).most_common():
    print(f"  {label:<10}{count}")

agreed = sum(1 for c in conflicts if c["predicted"] == c["title_declares"])
print(f"\n{len(conflicts)}/{len(worst)} confident errors have a title that declares "
      f"a different class than the label")
print(f"of those, the model agreed with the title in {agreed} cases")

pd.DataFrame(conflicts)[["true", "title_declares", "predicted", "title"]]

true label of the 25 most confident mistakes:
  question  14
  feature   6
  bug       5

9/25 confident errors have a title that declares a different class than the label
of those, the model agreed with the title in 8 cases


,true,title_declares,predicted,title
0,question,bug,feature,`sendtoaddress` tries to use unspendable UTXOs...
1,question,feature,feature,Allow webview context menus triggered by prima...
2,question,bug,bug,Error when runnning tensorflow.python.ops.spar...
3,feature,question,question,How to use opencv to erase text from images an...
4,question,bug,bug,ANN_MLP - large dataset results in stack overf...
5,question,feature,feature,please allow for multiple tunnels on same machine
6,question,feature,feature,[Feature Request] Add workbench action to spli...
7,bug,question,question,I need TensorFlow 2.2.0 but it is removed how ...
8,question,bug,bug,tf.numpy_function in tf.data.Dataset.map cause...


Two things fall out of this.

**`question` is where the model breaks down.** Fourteen of the twenty-five most
confident errors have `question` as their true label, against six for `feature`
and five for `bug`. That matches its consistently lowest per-class F1.

**About a third of confident errors look like label noise.** Nine of the
twenty-five have a title that explicitly declares a different type than the
ground-truth label — an issue titled `[Feature Request] Add workbench action to
split editor terminal below` is labelled `question` — and in most of those the
model's prediction agrees with the title rather than the label.

These labels come from maintainers applying project conventions, not from an
annotation protocol with adjudication. A ceiling below 1.0 is therefore built
into the dataset, and part of the remaining gap to any model is unreachable.
This belongs in threats to validity, and it is worth noting that the published
baseline faces exactly the same ceiling.

One further observation from scanning the list: at least one confident error is
an issue written in Portuguese. The dataset is not language-filtered, but this
is a marginal effect — one case in twenty-five — not a major error source.

In [9]:
latex = to_latex(
    board,
    caption=(
        "Per-repository and cross-repository $F_1$ for every model, "
        "against the NLBSE'24 SetFit baseline."
    ),
    label="final-leaderboard",
    path=TABLES / "final_leaderboard.tex",
)
print(latex[:600], "...")

\begin{table}[htbp]
  \centering
  \caption{Per-repository and cross-repository $F_1$ for every model, against the NLBSE'24 SetFit baseline.}
  \label{tab:final-leaderboard}
\begin{tabular}{lrrrrrrrr}
\toprule
 & react & tensorflow & vscode & bitcoin & opencv & overall & AUC & vs SetFit \\
model &  &  &  &  &  &  &  &  \\
\midrule
SetFit (NLBSE'24 baseline) & 0.8718 & 0.8644 & 0.8262 & 0.7555 & 0.8173 & 0.8270 & -- & 0.0000 \\
Ensemble (SetFit + TF-IDF + MPNet) & 0.8505 & 0.8520 & 0.7957 & 0.7691 & 0.8168 & 0.8168 & 0.9319 & -0.0102 \\
SetFit (MPNet) & 0.8438 & 0.8710 & 0.8104 & 0.7488 & 0.777 ...


---
## 4. Summary

| Finding | Evidence |
|---|---|
| Best result 0.8168, a soft-voting ensemble | 98.5% of the distance from a majority classifier to the baseline |
| Ensembles beat the baseline on `bitcoin` | 0.7691 and 0.7694 vs 0.7555 |
| SetFit-MPNet beats the baseline on `tensorflow` | 0.8710 vs 0.8644 |
| Contrastive fine-tuning is worth +0.0759 | matched settings, MiniLM 0.7816 vs frozen 0.7057 |
| A larger encoder is worth +0.0286 | matched settings, MPNet 0.8102 vs MiniLM 0.7816 |
| Encoder and fine-tuning are sub-additive | encoder keeps 57% of its frozen +0.0500 |
| Training settings matter as much as encoder size | reduced batch/seq cost 0.0166 |
| An additive projection failed by 0.038 | predicted 0.8482, measured 0.8102 |
| Ensembling beats every individual model | +0.0066 over its best member; best AUC 0.9319 |
| Neural and SetFit runs vary by ~0.01–0.015 | CNN moved 0.0140 across environments |
| Aggressive cleaning hurts | `full` scores 0.015 below `light`, consistently |
| Per-project training beats global | +0.068 on all five, with a fifth of the data |
| `bug` vs `question` is the dominant confusion | >25% of all errors |
| ~⅓ of confident errors look like label noise | title declares a different type than the label |

### Threats to validity worth carrying into the report

Every result was regenerated from a clean checkout on one machine.
**Twenty-one of twenty-two models returned bit-identical scores.**

The exception is informative rather than worrying. `SetFit (MPNet)` moved its
cross-repository F1 by 0.00004, while its per-repository scores moved by up to
0.0099 — `vscode` fell 0.0099 and `opencv` rose 0.0095, largely cancelling.
**The competition's averaged metric is roughly an order of magnitude more
stable than any single project's figure**, because it averages five
independent classifiers. Quote per-repository numbers to show *where* models
differ, not to claim one beats another by a small margin on one project.

Across *devices* the picture is different: the CNN scores 0.7578 on a CPU and
0.7438 on a GPU from identical code and seed, because the two backends use
different kernels and accumulation orders. Report a neural result with the
device that produced it.

Two earlier claims in this analysis did not survive being checked — a cuDNN
nondeterminism diagnosis whose fix changed nothing, and a SetFit variance
estimate that turned out to span a code change. Both are documented in
`code_overview.md` §7.3.